Evaluación Comparativa de Modelos


### Objetivo
Evaluar formalmente los modelos base en el conjunto de prueba:


## 1. Importación de Librerías

In [ ]:
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error, r2_score
)

sns.set_theme(style="whitegrid")
print("Librerías importadas correctamente.")

## 2. Cargar el Dataset Limpio
Sube el archivo `dataset_clientes_limpio.csv` que descargaste del Notebook 2.

In [ ]:
!wget https://raw.githubusercontent.com/FIJI-1919/TRABAJO_PROGRAMACION_CIENCIA/refs/heads/main/data/dataset_clientes_limpio.csv

In [ ]:
data_limpio = pd.read_csv("dataset_clientes_limpio.csv")

## 3. Funciones de Evaluación

Definimos aquí todas las funciones para calcular métricas y generar gráficos.

In [ ]:
def evaluate_classification(y_true, y_pred, y_prob=None) -> dict:
    """
    Calcula métricas de evaluación para clasificación binaria.
    Retorna: Accuracy, Precision, Recall, F1-Score y ROC-AUC.
    """
    metricas = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, zero_division=0),
    }
    if y_prob is not None:
        metricas["ROC-AUC"] = roc_auc_score(y_true, y_prob)
    return metricas


def evaluate_regression(y_true, y_pred) -> dict:
    """
    Calcula métricas de evaluación para regresión.
    Retorna: MAE, MSE, RMSE, R².
    """
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "MSE":  mse,
        "RMSE": np.sqrt(mse),
        "R²":   r2_score(y_true, y_pred),
    }


def plot_confusion_matrix(y_true, y_pred, nombre_modelo):
    """Grafica la matriz de confusión de un clasificador."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(cm, display_labels=["No Abandona", "Abandona"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Matriz de Confusión — {nombre_modelo}", fontweight="bold")
    plt.tight_layout()
    plt.show()


def plot_roc_curves(modelos, X_test, y_test):
    """Grafica las curvas ROC de todos los clasificadores en un solo eje."""
    fig, ax = plt.subplots(figsize=(8, 6))
    colores = ["steelblue", "coral", "mediumseagreen"]

    for (nombre, modelo), color in zip(modelos.items(), colores):
        y_prob = modelo.predict_proba(X_test)[:, 1] if hasattr(modelo, "predict_proba") else modelo.decision_function(X_test)
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f"{nombre} (AUC = {auc:.3f})", color=color, lw=2)

    ax.plot([0, 1], [0, 1], "k--", label="Clasificador aleatorio")
    ax.set_xlabel("Tasa de Falsos Positivos")
    ax.set_ylabel("Tasa de Verdaderos Positivos")
    ax.set_title("Curvas ROC — Comparación de Clasificadores", fontsize=13, fontweight="bold")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


def plot_regression_predictions(y_true, y_pred, nombre_modelo):
    """Grafica Valores Reales vs. Predichos para un regresor."""
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(y_true, y_pred, alpha=0.4, color="steelblue", s=15)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "r--", lw=1.5, label="Predicción perfecta")
    ax.set_xlabel("Valores Reales")
    ax.set_ylabel("Valores Predichos")
    ax.set_title(f"Real vs. Predicho — {nombre_modelo}", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_residuals(y_true, y_pred, nombre_modelo):
    """Grafica análisis de residuos para verificar homocedasticidad."""
    residuos = np.array(y_true) - np.array(y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].scatter(y_pred, residuos, alpha=0.4, color="coral", s=15)
    axes[0].axhline(0, color="black", linestyle="--", lw=1.5)
    axes[0].set_xlabel("Valores Predichos")
    axes[0].set_ylabel("Residuos")
    axes[0].set_title(f"Residuos vs. Predichos — {nombre_modelo}", fontweight="bold")

    axes[1].hist(residuos, bins=40, color="mediumseagreen", edgecolor="white")
    axes[1].set_xlabel("Residuo")
    axes[1].set_ylabel("Frecuencia")
    axes[1].set_title(f"Distribución de Residuos — {nombre_modelo}", fontweight="bold")

    plt.tight_layout()
    plt.show()


print("Funciones de evaluación definidas.")

## 4. Definición de Modelos

In [ ]:
def get_classification_models() -> dict:
    return {
        "LogisticRegression": Pipeline(steps=[
            ("classifier", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
        ]),
        "DecisionTreeClassifier": Pipeline(steps=[
            ("classifier", DecisionTreeClassifier(random_state=42, class_weight="balanced")),
        ]),
        "SVM": Pipeline(steps=[
            ("classifier", SVC(kernel="rbf", probability=True, random_state=42, class_weight="balanced")),
        ]),
    }

def get_regression_models() -> dict:
    return {
        "LinearRegression": Pipeline(steps=[("regressor", LinearRegression())]),
        "DecisionTreeRegressor": Pipeline(steps=[("regressor", DecisionTreeRegressor(random_state=42))]),
    }

def train_and_fit_model(pipeline, X_train, y_train):
    pipeline.fit(X_train, y_train)
    return pipeline

print("Modelos definidos.")

## 5. Particiones Train / Test

In [ ]:
SEMILLA = 42

X_clf = df.drop(columns=['abandono'])
y_clf = df['abandono']
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=SEMILLA, stratify=y_clf
)

X_reg = df.drop(columns=['gasto_mensual', 'abandono'])
y_reg = df['gasto_mensual']
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=SEMILLA
)

print("Entrenando modelos base...")
clasificadores = {n: train_and_fit_model(p, X_train_clf, y_train_clf) for n, p in get_classification_models().items()}
regresores     = {n: train_and_fit_model(p, X_train_reg, y_train_reg) for n, p in get_regression_models().items()}
print("Modelos listos para evaluación.")

## 6. Evaluación de Clasificadores

### ¿Por qué F1-Score como métrica principal?
El dataset presenta desbalance de clases (~40% abandono). En este contexto la **Accuracy** puede ser engañosa — un modelo que predice siempre "No abandona" tendría ~60%. El **F1-Score** balancea Precision y Recall penalizando errores en ambas clases.

### 6.1 Tabla de Métricas Comparativas

In [ ]:
resultados_clf = {}

for nombre, modelo in clasificadores.items():
    y_pred = modelo.predict(X_test_clf)
    y_prob = modelo.predict_proba(X_test_clf)[:, 1] if hasattr(modelo, 'predict_proba') else None
    resultados_clf[nombre] = evaluate_classification(y_test_clf, y_pred, y_prob)

df_res_clf = pd.DataFrame(resultados_clf).T
print("=== COMPARACIÓN DE CLASIFICADORES (conjunto de prueba) ===")
display(df_res_clf.round(4))

### 6.2 Matrices de Confusión

In [ ]:
for nombre, modelo in clasificadores.items():
    y_pred = modelo.predict(X_test_clf)
    plot_confusion_matrix(y_test_clf, y_pred, nombre)

### 6.3 Curvas ROC Comparativas

In [ ]:
plot_roc_curves(clasificadores, X_test_clf, y_test_clf)

## 7. Evaluación de Regresores

| Métrica | Descripción | Mejor valor |
|---|---|---|
| MAE | Error absoluto medio | Menor es mejor |
| MSE | Error cuadrático medio | Menor es mejor |
| RMSE | Raíz del MSE (misma unidad que target) | Menor es mejor |
| R² | Varianza explicada por el modelo | Cercano a 1 |

### 7.1 Tabla de Métricas Comparativas

In [ ]:
resultados_reg = {}

for nombre, modelo in regresores.items():
    y_pred = modelo.predict(X_test_reg)
    resultados_reg[nombre] = evaluate_regression(y_test_reg, y_pred)

df_res_reg = pd.DataFrame(resultados_reg).T
print("=== COMPARACIÓN DE REGRESORES (conjunto de prueba) ===")
display(df_res_reg.round(2))

### 7.2 Valores Reales vs. Predichos

In [ ]:
for nombre, modelo in regresores.items():
    y_pred = modelo.predict(X_test_reg)
    plot_regression_predictions(y_test_reg, y_pred, nombre)

### 7.3 Análisis de Residuos

In [ ]:
for nombre, modelo in regresores.items():
    y_pred = modelo.predict(X_test_reg)
    plot_residuals(y_test_reg, y_pred, nombre)

## 8. Conclusiones

### Clasificación
- Se comparan tres clasificadores: Regresión Logística, Árbol de Decisión y SVM.
- El **F1-Score** es la métrica prioritaria por el desbalance de clases.
- Las matrices de confusión revelan el trade-off entre falsos positivos (clientes retenidos innecesariamente) y falsos negativos (clientes perdidos sin intervención).

### Regresión
- El análisis de residuos permite verificar homocedasticidad — residuos sin patrón sistemático indican buen ajuste.
- El modelo con mayor R² y menor RMSE es el candidato a optimizar en el Notebook 4.

> **Próximo paso:** Optimización de hiperparámetros en el **Notebook 4**.